# Conditional Discrete-Palette DCGAN (cDCGAN) — Warm Start (18 Epochs)
### Palette-Conditioned Archetype Disentanglement ($k=4$) with Projection Discriminator & EMA

**Key Architectural Fixes & Training Recipe:**
1. **Provable Warm Start (G & D)**:
   - Transfers pretrained weights from the 50-epoch unconditional generator (`generator_weights_final.weights.h5`) by layer name.
   - Zero-initialized $4 \times 8,192$ class embedding in G guarantees $cG(z, \text{any class}) \equiv G(z)$ at step 0 (strictly asserted on 5 random latents, $\Delta < 10^{-5}$, no try/except).
   - Warm-starts Discriminator's 4 SpectralNormalization conv blocks + scalar score; zero-init projection embedding ensures $cD(x, \text{any class}) \equiv D(x)$ at step 0.
2. **Learning Rate Justification ($lr_D \le lr_G$, both $\le 1\times 10^{-4}$)**:
   - In the original run ($lr_G = 3\times 10^{-4}, lr_D = 1\times 10^{-4}$), Spectral Normalization made D so sample-efficient that by epoch 48, D loss dropped to $0.847$ while G loss climbed to $1.160$ (discriminator dominance).
   - To preserve generator diversity and allow the newly added projection weights to catch up without overpowering G, we default to $lr_G = 1\times 10^{-4}$ and $lr_D = 5\times 10^{-5}$ ($lr_D \le lr_G$, both $\le 1\times 10^{-4}$).
3. **Zero HuggingFace Dependency (Fast Local Ingestion)**:
   - Loads pre-extracted, preprocessed `sprites_42k.npy` (uint8 RGBA, 64x64) and `labels.npy` directly from the attached Kaggle dataset in $< 3$ seconds.
4. **Objective Cluster Nomenclature**:
   - Clusters are strictly labeled **Palette group 0**, **Palette group 1**, **Palette group 2**, and **Palette group 3** based on empirical K-means palette centroids.
5. **Exponential Moving Average (EMA)**:
   - Maintains generator shadow weights with decay $\alpha = 0.999$ to suppress mode oscillation.
6. **18-Epoch Multi-GPU Schedule**:
   - Total 5,922 updates (~20 minutes on Dual Tesla T4 GPUs), saving checkpoints every 3 epochs under `models/4_cDCGAN_k4_v1/`.


In [ ]:
# 1. Environment & Multi-GPU Strategy Setup
import os, sys, json, time, random, gc
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image as PILImage
import matplotlib.pyplot as plt

# Requirement 4: Print TensorFlow and Keras versions at start
print(f"Python Version     : {sys.version.split()[0]}")
print(f"TensorFlow Version : {tf.__version__}")
print(f"Keras Version      : {keras.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"Available GPUs     : {len(gpus)}")
for gpu in gpus:
    print(f"  -> {gpu}")

# MirroredStrategy for Dual Tesla T4 (or single GPU / CPU fallback)
if len(gpus) > 1:
    strategy = tf.distribute.MirroredStrategy()
    print(f"MirroredStrategy initialized across {strategy.num_replicas_in_sync} GPUs")
elif len(gpus) == 1:
    strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
    print("OneDeviceStrategy on single GPU")
else:
    strategy = tf.distribute.get_strategy()
    print("Default strategy (CPU fallback)")

NUM_REPLICAS = strategy.num_replicas_in_sync


In [ ]:
# 2. Hyperparameters & Directory Configuration
# Image & Architecture Constants
IMG_HEIGHT             = 64
IMG_WIDTH              = 64
CHANNELS               = 4
PALETTE_SIZE           = 32       # K = 32 discrete retro palette entries
LATENT_DIM             = 128
NUM_CLASSES            = 4        # Palette group 0, 1, 2, 3

G_CHANNELS             = [512, 256, 128, 64]
D_CHANNELS             = [64, 128, 256, 512]
G_INIT_SPATIAL         = 4

# ── Requirement 3: Learning Rate Configuration ──────────────────────────────
# In the 50-epoch unconditional run, G_LR was 3e-4 and D_LR was 1e-4.
# Spectral Normalization makes D very sample-efficient; higher D_LR causes D
# to overpower G. Here, both G and D are warm-started, so we require:
# lr_D <= lr_G, with both <= 1e-4.
G_LR                   = 1e-4     # Generator learning rate (<= 1e-4)
D_LR                   = 5e-5     # Discriminator learning rate (<= G_LR, <= 1e-4)
ADAM_BETA1             = 0.0      # Standard for Spectral Hinge GANs
ADAM_BETA2             = 0.9
EMA_DECAY              = 0.999    # Exponential moving average decay

# Schedule & Batching
TARGET_EPOCHS          = 18       # 18 Epochs total
CHECKPOINT_EVERY       = 3        # Checkpoint every 3 epochs
BATCH_SIZE_PER_REPLICA = 64
GLOBAL_BATCH_SIZE      = BATCH_SIZE_PER_REPLICA * NUM_REPLICAS
DIFFAUG_PAD            = 8

# Working Directories
BASE_DIR               = "/kaggle/working/models/4_cDCGAN_k4_v1"
CKPT_DIR               = f"{BASE_DIR}/checkpoints"
LOG_DIR                = f"{BASE_DIR}/logs"
SAMPLE_DIR             = f"{BASE_DIR}/samples"
EXPORT_DIR             = f"{BASE_DIR}/export"

for d in [BASE_DIR, CKPT_DIR, LOG_DIR, SAMPLE_DIR, EXPORT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Central Configuration Initialized:")
print(f"  Target Epochs      : {TARGET_EPOCHS} (Checkpoints every {CHECKPOINT_EVERY} epochs)")
print(f"  Global Batch Size  : {GLOBAL_BATCH_SIZE} ({BATCH_SIZE_PER_REPLICA}/GPU)")
print(f"  Learning Rates     : G_LR={G_LR:.1e}, D_LR={D_LR:.1e} (Justified: lr_D <= lr_G <= 1e-4)")
print(f"  EMA Decay          : {EMA_DECAY}")


In [ ]:
# 3. Empirical Palette & Straight-Through Gumbel-Softmax Synthesizer
KAGGLE_INPUT_DIR = "/kaggle/input/dcgan-pretrained-weights-alucard"
LOCAL_CONFIG_DIR = "config"

PALETTE_PATH = None
for p in [
    f"{KAGGLE_INPUT_DIR}/global_palette_32.npy",
    f"{LOCAL_CONFIG_DIR}/global_palette_32.npy",
    "models/3_DCGAN_42k_v1/config/global_palette_32.npy"
]:
    if os.path.exists(p):
        PALETTE_PATH = p
        break

if PALETTE_PATH is None:
    raise FileNotFoundError("global_palette_32.npy not found in input directories!")

global_palette_u8 = np.load(PALETTE_PATH)
global_palette_tf = tf.constant(global_palette_u8 / 255.0, dtype=tf.float32)
print(f"Loaded empirical 32-color palette: {global_palette_u8.shape}")

def logits_to_image(logits, palette_tf, tau=1.0, hard=True, noise=False):
    """
    Converts (B, 64, 64, 33) logits into discrete RGBA images in [-1, 1].
    Uses Straight-Through Gumbel-Softmax for hard discrete quantization with continuous gradients.
    """
    pl = logits[..., :PALETTE_SIZE]
    al = logits[..., PALETTE_SIZE:]
    
    if noise:
        u = tf.random.uniform(tf.shape(pl), minval=1e-6, maxval=1.0)
        gumbel_noise = -tf.math.log(-tf.math.log(u))
        pl = pl + gumbel_noise
        
    ys = tf.nn.softmax(pl / tau, axis=-1)
    if hard:
        y_hard = tf.one_hot(tf.argmax(ys, axis=-1), depth=PALETTE_SIZE, dtype=tf.float32)
        y = tf.stop_gradient(y_hard - ys) + ys
    else:
        y = ys
        
    rgb = tf.tensordot(y, palette_tf, axes=[[-1], [0]])
    a_s = tf.sigmoid(al)
    if hard:
        a_hard = tf.cast(a_s > 0.5, dtype=tf.float32)
        a = tf.stop_gradient(a_hard - a_s) + a_s
    else:
        a = a_s
        
    img_01 = tf.concat([rgb * a, a], axis=-1)
    return img_01 * 2.0 - 1.0


In [ ]:
# 4. Model Architectures: Conditional Generator & Projection Discriminator

# Unconditional Generator Builder (for exact weight loading)
def build_generator(latent_dim=LATENT_DIM):
    z_in = layers.Input(shape=(latent_dim,), name="latent_vector")
    x = layers.Dense(G_INIT_SPATIAL * G_INIT_SPATIAL * G_CHANNELS[0], use_bias=False, name="dense")(z_in)
    x = layers.BatchNormalization(name="batch_normalization")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu")(x)
    x = layers.Reshape((G_INIT_SPATIAL, G_INIT_SPATIAL, G_CHANNELS[0]), name="reshape")(x)
    x = layers.UpSampling2D(size=(2, 2), interpolation="nearest", name="up_sampling2d")(x)
    x = layers.Conv2D(G_CHANNELS[1], (3, 3), padding="same", use_bias=False, name="conv2d")(x)
    x = layers.BatchNormalization(name="batch_normalization_1")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu_1")(x)
    x = layers.UpSampling2D(size=(2, 2), interpolation="nearest", name="up_sampling2d_1")(x)
    x = layers.Conv2D(G_CHANNELS[2], (3, 3), padding="same", use_bias=False, name="conv2d_1")(x)
    x = layers.BatchNormalization(name="batch_normalization_2")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu_2")(x)
    x = layers.UpSampling2D(size=(2, 2), interpolation="nearest", name="up_sampling2d_2")(x)
    x = layers.Conv2D(64, (3, 3), padding="same", use_bias=False, name="conv2d_2")(x)
    x = layers.BatchNormalization(name="batch_normalization_3")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu_3")(x)
    x = layers.UpSampling2D(size=(2, 2), interpolation="nearest", name="up_sampling2d_3")(x)
    x = layers.Conv2D(64, (3, 3), padding="same", use_bias=False, name="conv2d_3")(x)
    x = layers.BatchNormalization(name="batch_normalization_4")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu_4")(x)
    x = layers.Conv2D(64, (3, 3), padding="same", name="conv2d_4")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu_5")(x)
    logits = layers.Conv2D(PALETTE_SIZE + 1, (3, 3), padding="same", name="palette_logits")(x)
    return keras.Model(z_in, logits, name="Discrete_DCGAN_Generator")

# Conditional Generator with Class Embedding
def build_cgenerator(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES):
    z_in = layers.Input(shape=(latent_dim,), name="latent_vector")
    y_in = layers.Input(shape=(), dtype="int32", name="class_label")
    dense_proj = layers.Dense(G_INIT_SPATIAL * G_INIT_SPATIAL * G_CHANNELS[0], use_bias=False, name="dense")(z_in)
    class_emb = layers.Embedding(num_classes, G_INIT_SPATIAL * G_INIT_SPATIAL * G_CHANNELS[0],
                                 embeddings_initializer="zeros", name="class_embedding")(y_in)
    x = layers.Add(name="add_class")([dense_proj, class_emb])
    x = layers.BatchNormalization(name="batch_normalization")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu")(x)
    x = layers.Reshape((G_INIT_SPATIAL, G_INIT_SPATIAL, G_CHANNELS[0]), name="reshape")(x)
    x = layers.UpSampling2D(size=(2, 2), interpolation="nearest", name="up_sampling2d")(x)
    x = layers.Conv2D(G_CHANNELS[1], (3, 3), padding="same", use_bias=False, name="conv2d")(x)
    x = layers.BatchNormalization(name="batch_normalization_1")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu_1")(x)
    x = layers.UpSampling2D(size=(2, 2), interpolation="nearest", name="up_sampling2d_1")(x)
    x = layers.Conv2D(G_CHANNELS[2], (3, 3), padding="same", use_bias=False, name="conv2d_1")(x)
    x = layers.BatchNormalization(name="batch_normalization_2")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu_2")(x)
    x = layers.UpSampling2D(size=(2, 2), interpolation="nearest", name="up_sampling2d_2")(x)
    x = layers.Conv2D(64, (3, 3), padding="same", use_bias=False, name="conv2d_2")(x)
    x = layers.BatchNormalization(name="batch_normalization_3")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu_3")(x)
    x = layers.UpSampling2D(size=(2, 2), interpolation="nearest", name="up_sampling2d_3")(x)
    x = layers.Conv2D(64, (3, 3), padding="same", use_bias=False, name="conv2d_3")(x)
    x = layers.BatchNormalization(name="batch_normalization_4")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu_4")(x)
    x = layers.Conv2D(64, (3, 3), padding="same", name="conv2d_4")(x)
    x = layers.LeakyReLU(0.2, name="leaky_re_lu_5")(x)
    logits = layers.Conv2D(PALETTE_SIZE + 1, (3, 3), padding="same", name="palette_logits")(x)
    return keras.Model([z_in, y_in], logits, name="Conditional_Discrete_DCGAN_Generator")

# Unconditional Discriminator Builder
def build_discriminator():
    img_in = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), name="sprite_input")
    def sn_conv(filters, strides=2):
        conv = layers.Conv2D(filters, (4, 4), strides=strides, padding="same")
        return layers.SpectralNormalization(conv)
    x = sn_conv(D_CHANNELS[0])(img_in)
    x = layers.LeakyReLU(0.2)(x)
    x = sn_conv(D_CHANNELS[1])(x)
    x = layers.LeakyReLU(0.2)(x)
    x = sn_conv(D_CHANNELS[2])(x)
    x = layers.LeakyReLU(0.2)(x)
    x = sn_conv(D_CHANNELS[3])(x)
    x = layers.LeakyReLU(0.2)(x)
    feat = layers.Flatten()(x)
    out = layers.Dense(1, name="scalar_score")(feat)
    return keras.Model(img_in, out, name="Spectral_Discriminator")

# Projection Discriminator Builder
def build_projection_discriminator(num_classes=NUM_CLASSES):
    img_in = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), name="sprite_input")
    y_in = layers.Input(shape=(), dtype="int32", name="class_label")
    def sn_conv(filters, strides=2):
        conv = layers.Conv2D(filters, (4, 4), strides=strides, padding="same")
        return layers.SpectralNormalization(conv)
    x = sn_conv(D_CHANNELS[0])(img_in)
    x = layers.LeakyReLU(0.2)(x)
    x = sn_conv(D_CHANNELS[1])(x)
    x = layers.LeakyReLU(0.2)(x)
    x = sn_conv(D_CHANNELS[2])(x)
    x = layers.LeakyReLU(0.2)(x)
    x = sn_conv(D_CHANNELS[3])(x)
    x = layers.LeakyReLU(0.2)(x)
    feat = layers.Flatten(name="feat_flatten")(x)
    base_score = layers.Dense(1, name="scalar_score")(feat)
    y_embed = layers.Embedding(num_classes, 4 * 4 * 512, embeddings_initializer="zeros", name="proj_embedding")(y_in)
    proj_score = layers.Dot(axes=1, name="dot_projection")([feat, y_embed])
    out = layers.Add(name="total_score")([base_score, proj_score])
    return keras.Model([img_in, y_in], out, name="Projection_Spectral_Discriminator")

with strategy.scope():
    generator = build_cgenerator()
    generator_ema = build_cgenerator()
    discriminator = build_projection_discriminator()

print("Conditional Models Initialized under Strategy:")
print(f"  Generator Parameters      : {generator.count_params():,}")
print(f"  Discriminator Parameters  : {discriminator.count_params():,}")


In [ ]:
# 5. Requirement 1: Warm Start & Strict Equivalence Assertion (No try/except)
G_WEIGHTS_PATH = None
D_WEIGHTS_PATH = None
for base_p in [KAGGLE_INPUT_DIR, "models/3_DCGAN_42k_v1/game_ready_assets", "game_ready_assets"]:
    g_cand = f"{base_p}/generator_weights_final.weights.h5"
    d_cand = f"{base_p}/discriminator_weights_final.weights.h5"
    if os.path.exists(g_cand) and os.path.exists(d_cand):
        G_WEIGHTS_PATH = g_cand
        D_WEIGHTS_PATH = d_cand
        break

if G_WEIGHTS_PATH is None or D_WEIGHTS_PATH is None:
    raise FileNotFoundError("Pretrained weights (.weights.h5) not found! Ensure input dataset is attached.")

print(f"Loading original G from {G_WEIGHTS_PATH}...")
orig_g = build_generator()
orig_g.load_weights(G_WEIGHTS_PATH)

print("Copying weights into cG (and cG_EMA) by layer name...")
copied_g = 0
for layer in orig_g.layers:
    if len(layer.weights) > 0:
        w = layer.get_weights()
        generator.get_layer(layer.name).set_weights(w)
        generator_ema.get_layer(layer.name).set_weights(w)
        copied_g += 1

print(f"  -> Successfully transferred {copied_g} layers into cG and cG_EMA.")

# Strict Assertion: cG(z, any class) == G(z) on 5 random z (max diff < 1e-5). Raise on failure, no try/except.
rng_check = np.random.RandomState(1337)
max_g_diff = 0.0
for trial in range(5):
    z_test = rng_check.normal(0, 1.0, size=(1, LATENT_DIM)).astype(np.float32)
    g_baseline = orig_g(z_test, training=False).numpy()
    for c in range(NUM_CLASSES):
        c_test = np.array([c], dtype=np.int32)
        cg_output = generator([z_test, c_test], training=False).numpy()
        diff = float(np.max(np.abs(g_baseline - cg_output)))
        max_g_diff = max(max_g_diff, diff)
        assert diff < 1e-5, f"Warm-start assertion failed at trial {trial}, class {c}: diff {diff} >= 1e-5"

print(f"[ASSERTION PASSED]: cG(z, any class) == G(z) across 5 random z and all classes!")
print(f"  Max G difference: {max_g_diff:.10f} (< 1e-5)")

# Discriminator Warm-Start Transfer
print(f"Loading original D from {D_WEIGHTS_PATH}...")
orig_d = build_discriminator()
orig_d.load_weights(D_WEIGHTS_PATH)

orig_sn = [l for l in orig_d.layers if isinstance(l, layers.SpectralNormalization)]
cd_sn = [l for l in discriminator.layers if isinstance(l, layers.SpectralNormalization)]
assert len(orig_sn) == len(cd_sn) == 4, "Discriminator SN block mismatch!"

for o_l, c_l in zip(orig_sn, cd_sn):
    c_l.set_weights(o_l.get_weights())
discriminator.get_layer('scalar_score').set_weights(orig_d.get_layer('scalar_score').get_weights())

# Strict Assertion for D: cD(x, any class) == D(x) on 5 random inputs (max diff < 1e-5)
x_test = rng_check.normal(0, 1.0, size=(5, IMG_HEIGHT, IMG_WIDTH, CHANNELS)).astype(np.float32)
d_baseline = orig_d(x_test, training=False).numpy()
max_d_diff = 0.0
for c in range(NUM_CLASSES):
    c_test = np.full((5,), c, dtype=np.int32)
    cd_output = discriminator([x_test, c_test], training=False).numpy()
    diff = float(np.max(np.abs(d_baseline - cd_output)))
    max_d_diff = max(max_d_diff, diff)
    assert diff < 1e-5, f"Discriminator warm-start assertion failed at class {c}: diff {diff} >= 1e-5"

print(f"[ASSERTION PASSED]: cD(x, any class) == D(x) across 5 random inputs!")
print(f"  Max D difference: {max_d_diff:.10f} (< 1e-5)")
print("  Status: Discriminator is warm-started (4 SpectralNormalization blocks + scalar score transferred).")

del orig_g, orig_d
gc.collect()


In [ ]:
# 6. Requirement 4: Fast Ingestion from Kaggle Dataset (No HuggingFace Download)
SPRITES_PATH = None
LABELS_PATH = None

for p in [KAGGLE_INPUT_DIR, "Dataset", "config"]:
    s_cand = f"{p}/sprites_42k.npy"
    l_cand = f"{p}/labels.npy"
    if os.path.exists(s_cand) and os.path.exists(l_cand):
        SPRITES_PATH = s_cand
        LABELS_PATH = l_cand
        break

if SPRITES_PATH is None:
    raise FileNotFoundError("sprites_42k.npy and labels.npy not found in dataset paths!")

print(f"Loading preprocessed sprites from: {SPRITES_PATH}")
t0 = time.time()
sprites_42k = np.load(SPRITES_PATH)
labels_42k = np.load(LABELS_PATH)
print(f"Loaded {len(sprites_42k):,} sprites in {time.time() - t0:.2f}s!")
print(f"  Sprites array shape : {sprites_42k.shape}, dtype={sprites_42k.dtype} ({sprites_42k.nbytes / (1024**2):.1f} MB)")
print(f"  Labels array shape  : {labels_42k.shape}, dtype={labels_42k.dtype}")

# Requirement 5: Display Palette group 0-3 Distribution
group_counts = np.bincount(labels_42k, minlength=NUM_CLASSES)
print("\nPalette Group Distribution:")
for c in range(NUM_CLASSES):
    pct = (group_counts[c] / len(labels_42k)) * 100.0
    print(f"  Palette group {c} : {group_counts[c]:,} sprites ({pct:.2f}%)")

# Distributed tf.data Pipeline
def preprocess_batch(x_uint8, y_cls):
    x_norm = (tf.cast(x_uint8, tf.float32) / 127.5) - 1.0
    return x_norm, y_cls

tf_ds = (
    tf.data.Dataset.from_tensor_slices((sprites_42k, labels_42k))
    .shuffle(10000, seed=42)
    .batch(GLOBAL_BATCH_SIZE, drop_remainder=True)
    .map(preprocess_batch, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

dist_dataset = strategy.experimental_distribute_dataset(tf_ds)
STEPS_PER_EPOCH = len(sprites_42k) // GLOBAL_BATCH_SIZE
print(f"\ntf.data Distributed Pipeline Ready: {STEPS_PER_EPOCH:,} steps/epoch at batch size {GLOBAL_BATCH_SIZE}")


In [ ]:
# 7. DiffAugment, TTUR Optimizers & EMA Distributed Training Step
with strategy.scope():
    aug_layer = layers.RandomTranslation(
        height_factor=(-DIFFAUG_PAD / float(IMG_HEIGHT), DIFFAUG_PAD / float(IMG_HEIGHT)),
        width_factor=(-DIFFAUG_PAD / float(IMG_WIDTH), DIFFAUG_PAD / float(IMG_WIDTH)),
        fill_mode="constant",
        fill_value=-1.0,
        interpolation="nearest"
    )

    # Justified learning rates: lr_D <= lr_G <= 1e-4
    g_optimizer = keras.optimizers.Adam(learning_rate=G_LR, beta_1=ADAM_BETA1, beta_2=ADAM_BETA2)
    d_optimizer = keras.optimizers.Adam(learning_rate=D_LR, beta_1=ADAM_BETA1, beta_2=ADAM_BETA2)
    
    # Pre-build optimizer slot variables under strategy scope to prevent placeholder uninitialized graph crash
    g_optimizer.build(generator.trainable_variables)
    d_optimizer.build(discriminator.trainable_variables)

def update_ema_weights(ema_model, source_model, decay=EMA_DECAY):
    for ema_var, src_var in zip(ema_model.weights, source_model.weights):
        ema_var.assign(decay * ema_var + (1.0 - decay) * src_var)

# Pure Python replica step (NO @tf.function here - inner tf.function inside strategy.run causes placeholder crash)
def train_step_per_replica(real_x, real_y):
    batch_size = tf.shape(real_x)[0]
    z = tf.random.normal((batch_size, LATENT_DIM))
    gen_y = tf.random.uniform((batch_size,), minval=0, maxval=NUM_CLASSES, dtype=tf.int32)
    
    with tf.GradientTape() as d_tape, tf.GradientTape() as g_tape:
        # 1. Synthesize fake images
        fake_logits = generator([z, gen_y], training=True)
        fake_x = logits_to_image(fake_logits, global_palette_tf, tau=1.0, hard=True, noise=True)
        
        # 2. DiffAugment translation
        real_x_aug = aug_layer(real_x, training=True)
        fake_x_aug = aug_layer(fake_x, training=True)
        
        # 3. Projection Discriminator scores
        d_real = discriminator([real_x_aug, real_y], training=True)
        d_fake = discriminator([fake_x_aug, gen_y], training=True)
        
        # 4. Geometric Hinge Loss
        d_loss_real = tf.reduce_mean(tf.nn.relu(1.0 - d_real))
        d_loss_fake = tf.reduce_mean(tf.nn.relu(1.0 + d_fake))
        d_loss = (d_loss_real + d_loss_fake) / float(NUM_REPLICAS)
        
        # Generator Loss
        g_loss = -tf.reduce_mean(d_fake) / float(NUM_REPLICAS)
        
    d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
    d_optimizer.apply_gradients(zip(d_grads, discriminator.trainable_variables))
    
    g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
    g_optimizer.apply_gradients(zip(g_grads, generator.trainable_variables))
    
    return d_loss * NUM_REPLICAS, g_loss * NUM_REPLICAS

# Only the outer multi-GPU orchestration is traced with @tf.function
@tf.function
def distributed_train_step(dist_batch):
    real_x, real_y = dist_batch
    per_d, per_g = strategy.run(train_step_per_replica, args=(real_x, real_y))
    d_loss = strategy.reduce(tf.distribute.ReduceOp.MEAN, per_d, axis=None)
    g_loss = strategy.reduce(tf.distribute.ReduceOp.MEAN, per_g, axis=None)
    return d_loss, g_loss

print("Optimizers pre-built successfully.")
print("Executing 1-step warmup across GPU replicas to trace graph...")
dummy_ds = tf.data.Dataset.from_tensors((
    tf.zeros((GLOBAL_BATCH_SIZE, IMG_HEIGHT, IMG_WIDTH, CHANNELS), dtype=tf.float32),
    tf.zeros((GLOBAL_BATCH_SIZE,), dtype=tf.int32)
))
dist_dummy = strategy.experimental_distribute_dataset(dummy_ds)
w_d, w_g = distributed_train_step(next(iter(dist_dummy)))
print(f"Warmup SUCCESS! All replica optimizer states initialized (d_loss={float(w_d):.4f}, g_loss={float(w_g):.4f}).")


In [ ]:
# 8. Conditioned Evaluation Grid (Palette group 0 - 3)
N_COLS = 8
eval_z = np.random.RandomState(1337).randn(N_COLS, LATENT_DIM).astype(np.float32)
eval_z_tile = np.tile(eval_z, (NUM_CLASSES, 1))
eval_y_tile = np.repeat(np.arange(NUM_CLASSES), N_COLS).astype(np.int32)

eval_z_tf = tf.constant(eval_z_tile)
eval_y_tf = tf.constant(eval_y_tile)

# Requirement 5: Objective naming
group_titles = [
    "Palette group 0",
    "Palette group 1",
    "Palette group 2",
    "Palette group 3"
]

def save_conditioned_samples(epoch, model_to_use, suffix=""):
    logits = model_to_use([eval_z_tf, eval_y_tf], training=False)
    imgs = logits_to_image(logits, global_palette_tf, tau=0.8, hard=True, noise=False).numpy()
    
    fig, axes = plt.subplots(NUM_CLASSES, N_COLS, figsize=(14, 7))
    for r in range(NUM_CLASSES):
        for c in range(N_COLS):
            idx = r * N_COLS + c
            img_data = (imgs[idx] + 1.0) * 0.5
            img_rgba = np.clip(img_data * 255.0, 0, 255).astype(np.uint8)
            ax = axes[r, c]
            ax.imshow(img_rgba)
            ax.axis('off')
            if c == 0:
                ax.set_title(group_titles[r], fontsize=10, loc='left', color='#1e3a8a', fontweight='bold')
                
    plt.suptitle(f"cDCGAN Conditioned Grid — Epoch {epoch:02d} {suffix}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    out_path = f"{SAMPLE_DIR}/cDCGAN_epoch_{epoch:02d}{suffix}.png"
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"  -> Saved sample grid to {out_path}")


In [ ]:
# 9. Production 18-Epoch Training Loop with Checkpoints Every 3 Epochs
print("=" * 80)
print(f"STARTING cDCGAN TRAINING LOOP — {TARGET_EPOCHS} EPOCHS TOTAL")
print(f"Steps per Epoch: {STEPS_PER_EPOCH:,} | Total Updates: {TARGET_EPOCHS * STEPS_PER_EPOCH:,}")
print("=" * 80)

# Step-0 evaluation grid
save_conditioned_samples(0, generator, suffix="_step0")

history_csv = f"{LOG_DIR}/cDCGAN_k4_training_history.csv"
with open(history_csv, "w") as f:
    f.write("epoch,d_loss,g_loss,epoch_time\n")

for epoch in range(1, TARGET_EPOCHS + 1):
    t_start = time.time()
    d_loss_accum = tf.constant(0.0)
    g_loss_accum = tf.constant(0.0)
    
    for step, batch in enumerate(dist_dataset):
        d_l, g_l = distributed_train_step(batch)
        d_loss_accum += d_l
        g_loss_accum += g_l
        
        # Exponential Moving Average update
        update_ema_weights(generator_ema, generator, decay=EMA_DECAY)
        
    avg_d = float(d_loss_accum.numpy()) / STEPS_PER_EPOCH
    avg_g = float(g_loss_accum.numpy()) / STEPS_PER_EPOCH
    elapsed = time.time() - t_start
    
    print(f"Epoch {epoch:02d}/{TARGET_EPOCHS} | D_Loss: {avg_d:.4f} | G_Loss: {avg_g:.4f} | Elapsed: {elapsed:.1f}s")
    with open(history_csv, "a") as f:
        f.write(f"{epoch},{avg_d:.4f},{avg_g:.4f},{elapsed:.1f}\n")
        
    # Save sample grids from EMA generator
    save_conditioned_samples(epoch, generator_ema, suffix="_ema")
    
    # Checkpoint every 3 epochs under models/4_cDCGAN_k4_v1/checkpoints/
    if epoch % CHECKPOINT_EVERY == 0:
        ckpt_path_g = f"{CKPT_DIR}/cgenerator_epoch_{epoch:02d}.weights.h5"
        ckpt_path_ema = f"{CKPT_DIR}/cgenerator_ema_epoch_{epoch:02d}.weights.h5"
        ckpt_path_d = f"{CKPT_DIR}/cdiscriminator_epoch_{epoch:02d}.weights.h5"
        generator.save_weights(ckpt_path_g)
        generator_ema.save_weights(ckpt_path_ema)
        discriminator.save_weights(ckpt_path_d)
        print(f"  [CHECKPOINT] Saved epoch {epoch:02d} weights under {CKPT_DIR}/")

# Final weights export
final_g_path = f"{EXPORT_DIR}/cgenerator_weights_final.weights.h5"
final_ema_path = f"{EXPORT_DIR}/cgenerator_ema_weights_final.weights.h5"
final_d_path = f"{EXPORT_DIR}/cdiscriminator_weights_final.weights.h5"

generator.save_weights(final_g_path)
generator_ema.save_weights(final_ema_path)
discriminator.save_weights(final_d_path)

print("=" * 80)
print(f"cDCGAN Training Complete! Production weights exported to {EXPORT_DIR}")
print("=" * 80)
